<a href="https://colab.research.google.com/github/cedizone/CS501R/blob/main/AI%20Judge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#PHASE 1 — Problem Formalization

##Description

Phase 1 establishes the formal definition of the evaluation problem by specifying the relationship between inputs, outputs, and evaluation objectives. In this project, the task is defined as a structured mapping from unstructured textual data (scholarship essays) to a set of quantitative scores representing human-centered attributes, namely authenticity, socioeconomic need, and motivational intent.

This phase clarifies the scope of the system, determines the type of predictive task (e.g., multi-dimensional regression), and defines the criteria by which system performance will be evaluated. By explicitly formalizing the problem, this stage ensures that all subsequent components—data collection, annotation, modeling, and analysis—are aligned with a consistent and well-defined objective.

##Summary

Phase 1 defines the evaluation task by:

* specifying inputs (essay text) and outputs (multi-dimensional scores),
* formalizing subjective human judgments into measurable variables, and
* establishing the criteria for assessing system performance.

In [ ]:
from dataclasses import dataclass
from typing import Dict


@dataclass
class EssayInput:
    essay_id: str
    text: str


@dataclass
class EssayScore:
    authenticity: float
    need: float
    desire: float


class ScholarshipEvaluator:
    def __init__(self):
        self.score_range = (1, 5)
        self.criteria = ["authenticity", "need", "desire"]

    def evaluate(self, essay: EssayInput) -> EssayScore:
        return EssayScore(
            authenticity=0.0,
            need=0.0,
            desire=0.0
        )


if __name__ == "__main__":
    essay = EssayInput(
        essay_id="001",
        text="I grew up in a small village and had to walk miles to attend school..."
    )

    evaluator = ScholarshipEvaluator()
    result = evaluator.evaluate(essay)

    print(result)

EssayScore(authenticity=0.0, need=0.0, desire=0.0)


#PHASE 2 — Rubric Design
## Description

Phase 2 develops a structured evaluation rubric that translates abstract, human-centered qualities into clearly defined and measurable criteria. In this project, attributes such as authenticity, socioeconomic need, and motivational intent are operationalized through explicit indicators and scoring guidelines.

The rubric serves as the foundation for both human annotation and automated evaluation, ensuring consistency, interpretability, and reproducibility. By defining how each attribute is identified and scored within an essay, this phase reduces subjectivity and establishes a standardized framework that can be applied across different evaluators, including large language models.

##Summary

Phase 2 constructs the scoring framework by:

* defining measurable indicators for each evaluation dimension,
* establishing consistent scoring scales and guidelines, and
* creating a standardized rubric for both human and model-based evaluation.

In [ ]:
# ===============================
# PHASE 2: RUBRIC DESIGN
# ===============================

from dataclasses import dataclass
from typing import List, Dict


# -------------------------------
# Define Criteria Structure
# -------------------------------
@dataclass
class Criterion:
    name: str
    description: str
    indicators: List[str]
    score_range: tuple


# -------------------------------
# Build Rubric
# -------------------------------
class EvaluationRubric:
    def __init__(self):
        self.criteria = self._build_rubric()

    def _build_rubric(self) -> Dict[str, Criterion]:
        return {
            "authenticity": Criterion(
                name="authenticity",
                description="Degree to which the essay reflects genuine personal experience and honesty.",
                indicators=[
                    "Specific personal experiences",
                    "Consistency in narrative",
                    "Lack of exaggerated or generic claims"
                ],
                score_range=(1, 5)
            ),
            "need": Criterion(
                name="need",
                description="Extent of demonstrated socioeconomic hardship or financial need.",
                indicators=[
                    "Explicit mention of financial hardship",
                    "Family or personal economic challenges",
                    "Barriers to education"
                ],
                score_range=(1, 5)
            ),
            "desire": Criterion(
                name="desire",
                description="Strength of motivation, ambition, and commitment toward educational goals.",
                indicators=[
                    "Clear goals and aspirations",
                    "Evidence of effort or perseverance",
                    "Long-term vision"
                ],
                score_range=(1, 5)
            )
        }

    def get_criterion(self, name: str) -> Criterion:
        return self.criteria.get(name)

    def list_criteria(self) -> List[str]:
        return list(self.criteria.keys())


# -------------------------------
# Example Usage
# -------------------------------
if __name__ == "__main__":
    rubric = EvaluationRubric()

    for name, criterion in rubric.criteria.items():
        print(f"\n{name.upper()}")
        print("Description:", criterion.description)
        print("Indicators:", criterion.indicators)


AUTHENTICITY
Description: Degree to which the essay reflects genuine personal experience and honesty.
Indicators: ['Specific personal experiences', 'Consistency in narrative', 'Lack of exaggerated or generic claims']

NEED
Description: Extent of demonstrated socioeconomic hardship or financial need.
Indicators: ['Explicit mention of financial hardship', 'Family or personal economic challenges', 'Barriers to education']

DESIRE
Description: Strength of motivation, ambition, and commitment toward educational goals.
Indicators: ['Clear goals and aspirations', 'Evidence of effort or perseverance', 'Long-term vision']


#PHASE 3 — Dataset Specification
##Description

Phase 3 defines the formal structure and schema of the dataset used throughout the project. This includes specifying how raw essay data, associated metadata, and evaluation labels are organized, stored, and accessed.

The goal of this phase is to ensure consistency, scalability, and compatibility across all stages of the pipeline, including data preprocessing, annotation, model evaluation, and analysis. By establishing a clear data schema, this phase enables reproducible experimentation and reduces ambiguity in how inputs and outputs are handled within the system.

##Summary

Phase 3 establishes the dataset structure by:

* defining the fields and formats for essay data and metadata,
* specifying how evaluation labels are represented, and
* standardizing the data schema for consistent use across the pipeline.

In [ ]:
# ===============================
# PHASE 3: DATASET SPECIFICATION
# ===============================

from dataclasses import dataclass, asdict
from typing import Optional, Dict
import json


# -------------------------------
# Define Dataset Entry
# -------------------------------
@dataclass
class EssayRecord:
    essay_id: str
    text: str

    # Optional metadata
    country: Optional[str] = None
    age: Optional[int] = None
    gender: Optional[str] = None

    # Labels (to be filled later in Phase 6)
    authenticity: Optional[float] = None
    need: Optional[float] = None
    desire: Optional[float] = None


# -------------------------------
# Dataset Utility Functions
# -------------------------------
class DatasetManager:
    def __init__(self):
        self.dataset = []

    def add_record(self, record: EssayRecord):
        self.dataset.append(record)

    def save_to_json(self, filepath: str):
        with open(filepath, "w") as f:
            json.dump([asdict(r) for r in self.dataset], f, indent=4)

    def load_from_json(self, filepath: str):
        with open(filepath, "r") as f:
            data = json.load(f)
            self.dataset = [EssayRecord(**item) for item in data]

    def get_all_records(self):
        return self.dataset


# -------------------------------
# Example Usage
# -------------------------------
if __name__ == "__main__":
    manager = DatasetManager()

    sample = EssayRecord(
        essay_id="001",
        text="I grew up in a low-income household and worked part-time to support my family...",
        country="Ghana",
        age=18
    )

    manager.add_record(sample)
    manager.save_to_json("dataset.json")

    print("Dataset saved successfully.")

Dataset saved successfully.


#PHASE 4 — Data Acquisition & Integration (Kaggle Dataset)
##Description

Phase 4 focuses on acquiring and integrating a real-world dataset into the project pipeline. In this case, the dataset is sourced from a publicly available benchmark on Kaggle, specifically the Learning Agency Lab – Automated Essay Scoring 2.0 competition.

This phase involves downloading the dataset using the Kaggle API, extracting the relevant essay data, and transforming it into the standardized schema defined in Phase 3. The objective is to ensure that all raw data is properly structured and accessible for subsequent preprocessing, annotation, and evaluation.

By grounding the project in a real, large-scale dataset, this phase enhances the reliability, reproducibility, and empirical validity of the overall system.

##Summary

Phase 4 acquires and prepares real-world data by:

* downloading the dataset using the Kaggle API,
* extracting essay text and relevant fields, and
* converting the dataset into the standardized format defined earlier.

##Step 1 — Install Kaggle

In [ ]:
pip install kaggle

##Step 2 — Upload API Key

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"ebenezer2446","key":"2d4a263448546c23cb832235e75db803"}'}

##Step 3 — Configure Kaggle in Colab

In [ ]:
import os

os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

##Step 4 — Download Dataset

In [ ]:
!kaggle competitions download -c learning-agency-lab-automated-essay-scoring-2
!unzip learning-agency-lab-automated-essay-scoring-2.zip

100% 11.9M/11.9M [00:03<00:00, 4.14MB/s]

Archive:  learning-agency-lab-automated-essay-scoring-2.zip
  inflating: sample_submission.csv   
  inflating: test.csv                
  inflating: train.csv               


##Step 5 — Verify Files

In [ ]:
import os
os.listdir()

['.config',
 'learning-agency-lab-automated-essay-scoring-2.zip',
 'dataset.json',
 'train.csv',
 'test.csv',
 'sample_submission.csv',
 'sample_data']

In [ ]:
import pandas as pd
import uuid
from dataclasses import dataclass, asdict
import json


# -------------------------------
# Schema
# -------------------------------
@dataclass
class EssayRecord:
    essay_id: str
    text: str
    authenticity: float = None
    need: float = None
    desire: float = None


class DatasetManager:
    def __init__(self):
        self.dataset = []

    def add_record(self, record):
        self.dataset.append(record)

    def save(self, path):
        with open(path, "w") as f:
            json.dump([asdict(r) for r in self.dataset], f, indent=4)


# -------------------------------
# Load Data
# -------------------------------
df = pd.read_csv("train.csv")
print("Columns:", df.columns)


# -------------------------------
# Convert Data
# -------------------------------
manager = DatasetManager()

for _, row in df.iterrows():
    essay = row.get("full_text")

    if not isinstance(essay, str):
        continue

    essay = essay.strip()

    if len(essay) < 100:
        continue

    manager.add_record(
        EssayRecord(
            essay_id=str(uuid.uuid4()),
            text=essay
        )
    )


# -------------------------------
# Save Dataset
# -------------------------------
manager.save("dataset.json")

print(f"Saved {len(manager.dataset)} essays.")

Columns: Index(['essay_id', 'full_text', 'score'], dtype='object')
Saved 17307 essays.


#PHASE 5 — Data Cleaning & Intelligent Filtering
##Description

This phase refines the dataset by removing low-quality and irrelevant essays while prioritizing texts that exhibit personal narrative characteristics. The filtering process combines structural validation, linguistic normalization, and heuristic-based classification to distinguish between personal essays and purely academic or informational writing.

The objective is to align the dataset with the project’s evaluation goals by ensuring that retained essays contain meaningful signals related to human-centered attributes such as authenticity, socioeconomic need, and motivation.

##Summary

Phase 5 enhances dataset quality by:

* cleans and normalizes essay text,
* removes low-quality or invalid entries,
* filters out academic/informational essays,
* retains narrative-driven, human-centered content.

In [ ]:
# ===============================
# PHASE 5: DATA CLEANING & ADVANCED FILTERING (FINAL)
# ===============================

import pandas as pd
import uuid
import re
from dataclasses import dataclass, asdict
import json


# -------------------------------
# Schema
# -------------------------------
@dataclass
class EssayRecord:
    essay_id: str
    text: str
    authenticity: float = None
    need: float = None
    desire: float = None


class DatasetManager:
    def __init__(self):
        self.dataset = []

    def add_record(self, record):
        self.dataset.append(record)

    def save(self, path):
        with open(path, "w") as f:
            json.dump([asdict(r) for r in self.dataset], f, indent=4)


# -------------------------------
# Load Data
# -------------------------------
df = pd.read_csv("train.csv")


# -------------------------------
# BASIC CLEANING
# -------------------------------
def clean_text(text):
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    return text


def is_valid_essay(text):
    if not isinstance(text, str):
        return False

    text = text.strip()

    if len(text) < 200:
        return False

    if len(text) > 4000:
        return False

    if "." not in text:
        return False

    return True


# -------------------------------
# FILTER 1: First-person presence
# -------------------------------
def has_first_person(text):
    text_lower = text.lower()

    return (
        " i " in text_lower or
        text_lower.startswith("i ") or
        " my " in text_lower or
        " we " in text_lower
    )


# -------------------------------
# FILTER 2: Personal signals
# -------------------------------
def personal_signal_score(text):
    text_lower = text.lower()

    keywords = [
        "family", "life", "grew up", "experience",
        "struggle", "challenge", "difficult",
        "dream", "goal", "aspire",
        "my mother", "my father", "my parents"
    ]

    return sum(1 for k in keywords if k in text_lower)


# -------------------------------
# FILTER 3: Story structure
# -------------------------------
def has_story_structure(text):
    text_lower = text.lower()

    patterns = [
        "when i", "one day", "growing up",
        "i remember", "in my life", "there was a time"
    ]

    return any(p in text_lower for p in patterns)


# -------------------------------
# FILTER 4: Academic detection
# -------------------------------
def is_academic(text):
    text_lower = text.lower()

    academic_keywords = [
        "the author", "this essay", "this article",
        "in conclusion", "according to",
        "the text states", "evidence suggests",
        "research shows", "studies show",
        "this passage", "the passage"
    ]

    return any(k in text_lower for k in academic_keywords)


# -------------------------------
# FILTER 5: Heavy quoting (remove evidence essays)
# -------------------------------
def has_heavy_quoting(text):
    return text.count('"') >= 4


# -------------------------------
# FILTER 6: Argumentative tone
# -------------------------------
def is_argumentative(text):
    text_lower = text.lower()

    keywords = [
        "should", "must", "this shows",
        "this proves", "evidence", "therefore"
    ]

    return any(k in text_lower for k in keywords)


# -------------------------------
# FINAL FILTER LOGIC
# -------------------------------
def is_valid_personal_essay(text):
    if not has_first_person(text):
        return False

    if is_academic(text):
        return False

    if has_heavy_quoting(text):
        return False

    if is_argumentative(text):
        return False

    score = personal_signal_score(text)

    if score >= 3:
        return True

    if has_story_structure(text):
        return True

    return False


# -------------------------------
# APPLY PIPELINE
# -------------------------------
manager = DatasetManager()

for _, row in df.iterrows():
    essay = row.get("full_text")

    if not is_valid_essay(essay):
        continue

    essay = clean_text(essay)

    if not is_valid_personal_essay(essay):
        continue

    manager.add_record(
        EssayRecord(
            essay_id=str(uuid.uuid4()),
            text=essay
        )
    )


# -------------------------------
# SAVE RESULT
# -------------------------------
manager.save("clean_dataset.json")

print(f"Final dataset size: {len(manager.dataset)}")

Final dataset size: 427


#PHASE 6 — Data Annotation (Gold Labels)
##Description

This phase enhances the annotation process by integrating an LLM-based assistant to generate preliminary labels for each essay. Human verification is then used to confirm or adjust these labels, ensuring both efficiency and reliability. This hybrid approach reduces annotation time while maintaining high-quality ground truth data.

##Summary

Phase 6 creates labeled data by:

* applying the evaluation rubric to each essay,
* assigning numerical scores for each dimension, and
* producing a gold-standard dataset for evaluation and benchmarking.

##Step 1 — Install OpenAI SDK

In [ ]:
!pip install openai

##Step 2 — Load API Key from Colab Secrets

In [ ]:
from google.colab import userdata
import os

# Load secret
os.environ["OPENAI_API_KEY"] = userdata.get("OpenAI")

In [ ]:
# ===============================
# PHASE 6: AI-ASSISTED ANNOTATION (FINAL)
# ===============================

import json
from openai import OpenAI

client = OpenAI()


# -------------------------------
# Load Clean Dataset
# -------------------------------
with open("clean_dataset.json", "r") as f:
    data = json.load(f)


# -------------------------------
# AI Label Function
# -------------------------------
def get_ai_labels(text):
    prompt = f"""
You are evaluating a scholarship-style essay.

Score from 1 to 5:

1. Authenticity (personal, real experience)
2. Socioeconomic Need (financial hardship or life difficulty)
3. Desire (motivation, ambition, goals)

IMPORTANT RULES:
- If NOT personal → give LOW scores (1 or 2)
- Be strict and consistent

Return ONLY JSON:
{{
  "authenticity": number,
  "need": number,
  "desire": number
}}

Essay:
{text[:1200]}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content


# -------------------------------
# Annotation Loop
# -------------------------------
labeled_data = []

NUM_TO_LABEL = 30  # adjust if needed

for i, record in enumerate(data[:NUM_TO_LABEL]):
    print("\n" + "="*80)
    print(f"Essay {i+1}")
    print("="*80)

    print(record["text"][:800])  # preview

    # -------------------------------
    # Get AI Suggestion
    # -------------------------------
    ai_output = get_ai_labels(record["text"])
    print("\n🤖 AI Suggestion:", ai_output)

    try:
        ai_json = json.loads(ai_output)
    except:
        print("⚠️ AI output parsing failed, skipping...")
        continue

    # -------------------------------
    # Human Input (fast confirm/edit)
    # -------------------------------
    auth = input(f"Authenticity [{ai_json['authenticity']}]: ")
    need = input(f"Need [{ai_json['need']}]: ")
    des = input(f"Desire [{ai_json['desire']}]: ")

    # -------------------------------
    # Final Values (Human overrides AI)
    # -------------------------------
    final_auth = float(auth) if auth else ai_json["authenticity"]
    final_need = float(need) if need else ai_json["need"]
    final_des = float(des) if des else ai_json["desire"]

    # -------------------------------
    # Save BOTH AI + Human Labels
    # -------------------------------
    record["authenticity"] = final_auth
    record["need"] = final_need
    record["desire"] = final_des

    record["ai_authenticity"] = ai_json["authenticity"]
    record["ai_need"] = ai_json["need"]
    record["ai_desire"] = ai_json["desire"]

    labeled_data.append(record)


# -------------------------------
# Save Labeled Dataset
# -------------------------------
with open("labeled_dataset.json", "w") as f:
    json.dump(labeled_data, f, indent=4)

print(f"\n✅ Saved {len(labeled_data)} labeled essays.")


Essay 1
The 'Face' on Mars is just a hoax. It seems as though the mound on this planet is just a mesa. If you don't believe me, then listen to me very closely. The very similar face on the side of Mars can be very defining but not a single astronout have researched at the Mesa up close and personal. But even if it was made by aliens, it still wouldn't tally up right? The 'Face' was first noticed in 1976 by Viking 1 spacecraft, circling around Mars, after trying to find its sistership, Viking 2. As Viking 1 was circling around, it spotted a likeness to the human face. And the mound was found in Cydonia, a very well-know area on the Red Planet to house mesas. As the years went by, they looked more into the mound and Scientists concluded that it was just another mesa, but this moutain-like hill had s

🤖 AI Suggestion: {
  "authenticity": 1,
  "need": 1,
  "desire": 1
}
Authenticity [1]: 1
Need [1]: 2
Desire [1]: 2

Essay 2
Have you ever wanted to become a hero? Have you wanted to help pe

#PHASE 7 — Annotation Validation (FINAL)
##Description

This phase evaluates the reliability and consistency of the annotated dataset by analyzing both human-validated labels and AI-generated suggestions. It includes statistical summaries, distribution analysis, correlation checks, and agreement measurement between AI and human annotations.

The goal is to ensure that the dataset is consistent, unbiased, and suitable as a ground truth for model evaluation.

In [ ]:
# ===============================
# PHASE 7: ANNOTATION VALIDATION (FULL)
# ===============================

import json
import pandas as pd
import numpy as np


# -------------------------------
# Load Labeled Dataset
# -------------------------------
with open("labeled_dataset.json", "r") as f:
    data = json.load(f)

print(f"Loaded {len(data)} labeled essays.")


# ===============================
# 🔷 PART 1: DATA INTEGRITY VALIDATION (FIX + LOG)
# ===============================

corrections = []

for i, record in enumerate(data):
    for key in ["authenticity", "need", "desire"]:
        original = record[key]

        if original < 1 or original > 5:
            corrected = max(1, min(5, original))

            corrections.append({
                "essay_index": i,
                "field": key,
                "original": original,
                "corrected": corrected
            })

            record[key] = corrected

# Save cleaned dataset
with open("labeled_dataset_clean.json", "w") as f:
    json.dump(data, f, indent=4)

# Save correction log
with open("corrections_log.json", "w") as f:
    json.dump(corrections, f, indent=4)

print(f"\n✅ Fixed {len(corrections)} invalid values")

if len(data) > 0:
    print("Error rate:", round(len(corrections) / len(data), 4))


# ===============================
# 🔷 PART 2: STATISTICAL VALIDATION
# ===============================

df = pd.DataFrame(data)


# -------------------------------
# 1. Basic Statistics
# -------------------------------
print("\n=== HUMAN LABEL STATISTICS ===")
print(df[["authenticity", "need", "desire"]].describe())


# -------------------------------
# 2. Score Distribution
# -------------------------------
print("\n=== SCORE DISTRIBUTION ===")
for col in ["authenticity", "need", "desire"]:
    print(f"\n{col.upper()}")
    print(df[col].value_counts().sort_index())


# -------------------------------
# 3. Correlation Matrix
# -------------------------------
print("\n=== CORRELATION MATRIX (HUMAN) ===")
print(df[["authenticity", "need", "desire"]].corr())


# -------------------------------
# 4. Outlier Detection
# -------------------------------
def detect_outliers(series):
    mean = np.mean(series)
    std = np.std(series)
    return series[(series < mean - 2*std) | (series > mean + 2*std)]

print("\n=== OUTLIERS ===")
for col in ["authenticity", "need", "desire"]:
    outliers = detect_outliers(df[col])
    print(f"{col}: {len(outliers)} outliers")


# ===============================
# 🔷 PART 3: AI vs HUMAN AGREEMENT
# ===============================

print("\n=== AI vs HUMAN AGREEMENT ===")

for col in ["authenticity", "need", "desire"]:
    ai_col = f"ai_{col}"

    if ai_col in df.columns:
        diff = abs(df[col] - df[ai_col])

        print(f"\n{col.upper()}")
        print("Average difference:", round(diff.mean(), 3))
        print("Max difference:", diff.max())
    else:
        print(f"{ai_col} not found.")


# -------------------------------
# Exact Match Rate
# -------------------------------
print("\n=== EXACT MATCH RATE ===")

for col in ["authenticity", "need", "desire"]:
    ai_col = f"ai_{col}"

    if ai_col in df.columns:
        matches = (df[col] == df[ai_col]).sum()
        total = len(df)

        print(f"{col}: {matches}/{total} = {round(matches/total, 2)}")


# -------------------------------
# High Disagreement Samples
# -------------------------------
print("\n=== HIGH DISAGREEMENT SAMPLES ===")

threshold = 2

for col in ["authenticity", "need", "desire"]:
    ai_col = f"ai_{col}"

    if ai_col in df.columns:
        mask = abs(df[col] - df[ai_col]) >= threshold
        subset = df[mask]

        print(f"\n{col} high disagreement: {len(subset)} samples")

        if len(subset) > 0:
            print(subset[["text", col, ai_col]].head(2))


# ===============================
# 🔷 FINAL SUMMARY
# ===============================

print("\n=== SUMMARY ===")

print(f"Total labeled samples: {len(df)}")

avg_scores = df[["authenticity", "need", "desire"]].mean()
print("\nAverage Scores:")
print(avg_scores)

print("\n✅ Phase 7 validation complete.")

Loaded 21 labeled essays.

✅ Fixed 1 invalid values
Error rate: 0.0476

=== HUMAN LABEL STATISTICS ===
       authenticity       need     desire
count     21.000000  21.000000  21.000000
mean       1.523810   1.904762   2.428571
std        0.928388   0.436436   0.597614
min        1.000000   1.000000   2.000000
25%        1.000000   2.000000   2.000000
50%        1.000000   2.000000   2.000000
75%        2.000000   2.000000   3.000000
max        5.000000   3.000000   4.000000

=== SCORE DISTRIBUTION ===

AUTHENTICITY
authenticity
1.0    13
2.0     7
5.0     1
Name: count, dtype: int64

NEED
need
1.0     3
2.0    17
3.0     1
Name: count, dtype: int64

DESIRE
desire
2.0    13
3.0     7
4.0     1
Name: count, dtype: int64

=== CORRELATION MATRIX (HUMAN) ===
              authenticity      need    desire
authenticity      1.000000 -0.611131  0.386227
need             -0.611131  1.000000  0.164317
desire            0.386227  0.164317  1.000000

=== OUTLIERS ===
authenticity: 1 outliers
nee

#PHASE 8 — LLM Evaluation System (Core Engine)
##Description

Phase 8 implements the core evaluation system in which a large language model is used to automatically score essays based on the defined rubric. The system applies a standardized prompt to each essay and generates scores for authenticity, socioeconomic need, and motivational intent.

This phase operationalizes the evaluation framework developed earlier, enabling scalable and consistent scoring across the dataset. The outputs of this system will later be compared against human-labeled data to assess model alignment and performance.

Summary

Phase 8:

* applies an LLM to evaluate essays,
* generates structured scores using a consistent prompt, and
* produces model outputs for comparison with human labels.Summary


In [ ]:
# ===============================
# PHASE 8: LLM EVALUATION SYSTEM (FINAL UPGRADED)
# ===============================

# -------------------------------
# Install + Setup
# -------------------------------
!pip install openai

from google.colab import userdata
import os
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = userdata.get("OpenAI")
client = OpenAI()

# -------------------------------
# Imports
# -------------------------------
import json
import re
import time


# -------------------------------
# Load Dataset
# -------------------------------
with open("labeled_dataset_clean.json", "r") as f:
    data = json.load(f)

print(f"Loaded {len(data)} essays for evaluation.")


# -------------------------------
# JSON Extraction (ROBUST)
# -------------------------------
def extract_json(text):
    try:
        return json.loads(text)
    except:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
    return None


# -------------------------------
# LLM Evaluation Function (WITH RETRY)
# -------------------------------
def evaluate_with_llm(text, retries=3):
    prompt = f"""
You are an expert scholarship evaluator.

Evaluate the essay and score from 1 to 5:

1. Authenticity (real personal experience)
2. Socioeconomic Need (financial/life hardship)
3. Desire (motivation and ambition)

SCORING RULES:
- Non-personal / academic → score 1–2
- Weak personal → score 2–3
- Strong personal narrative → score 4–5

IMPORTANT:
- Be strict
- Be consistent
- Do NOT inflate scores

Return ONLY JSON:
{{
  "authenticity": number,
  "need": number,
  "desire": number
}}

Essay:
{text[:1200]}
"""

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=0
            )

            return response.choices[0].message.content

        except Exception as e:
            print(f"Retry {attempt+1} failed:", e)
            time.sleep(2)

    return None


# -------------------------------
# Run Evaluation (SAFE LOOP)
# -------------------------------
evaluated_data = []

for i, record in enumerate(data):
    print("\n" + "="*80)
    print(f"Essay {i+1}/{len(data)}")
    print("="*80)

    print("\n📄 Essay Preview:")
    print(record["text"][:500])

    output = evaluate_with_llm(record["text"])

    if output is None:
        print("⚠️ Failed after retries, skipping...")
        continue

    print("\n🤖 RAW LLM OUTPUT:")
    print(output)

    result = extract_json(output)

    if result is None:
        print("⚠️ Could not parse JSON, skipping...")
        continue

    # -------------------------------
    # Save results safely
    # -------------------------------
    record["llm_authenticity"] = result.get("authenticity", None)
    record["llm_need"] = result.get("need", None)
    record["llm_desire"] = result.get("desire", None)

    print("\n✅ Parsed Result:")
    print(result)

    evaluated_data.append(record)

    # -------------------------------
    # SAVE PROGRESS EVERY 5 STEPS
    # -------------------------------
    if i % 5 == 0:
        with open("evaluated_dataset_backup.json", "w") as f:
            json.dump(evaluated_data, f, indent=4)


# -------------------------------
# Final Save
# -------------------------------
with open("evaluated_dataset.json", "w") as f:
    json.dump(evaluated_data, f, indent=4)

print("\n✅ LLM evaluation complete.")

Loaded 21 essays for evaluation.

Essay 1/21

📄 Essay Preview:
The 'Face' on Mars is just a hoax. It seems as though the mound on this planet is just a mesa. If you don't believe me, then listen to me very closely. The very similar face on the side of Mars can be very defining but not a single astronout have researched at the Mesa up close and personal. But even if it was made by aliens, it still wouldn't tally up right? The 'Face' was first noticed in 1976 by Viking 1 spacecraft, circling around Mars, after trying to find its sistership, Viking 2. As Vikin

🤖 RAW LLM OUTPUT:
```json
{
  "authenticity": 1,
  "need": 1,
  "desire": 1
}
```

✅ Parsed Result:
{'authenticity': 1, 'need': 1, 'desire': 1}

Essay 2/21

📄 Essay Preview:
Have you ever wanted to become a hero? Have you wanted to help people through tough times? Well I know the job just for you! A Seagoing Cowboy. My name is Luke and I was a seagoing cowboy. One day my friend invited me to go to Europe on a cattle boat. I knew I 

#PHASE 9 — Prompt Engineering & Optimization
## Description

Phase 9 focuses on improving the performance and reliability of the LLM evaluation system by systematically designing and refining prompts. Different prompt formulations can significantly impact how the model interprets and scores essays.

This phase involves creating multiple prompt variants, testing them across the dataset, and comparing their outputs against human-labeled ground truth. The goal is to identify a prompt that produces the most accurate, consistent, and aligned evaluations.

##Summary

Phase 9:

designs multiple prompt strategies,
evaluates their performance, and
selects the best-performing prompt based on alignment with human labels.

In [ ]:
# ===============================
# PHASE 9: PROMPT ENGINEERING
# ===============================

import json
from openai import OpenAI
from google.colab import userdata
import os

# Setup
os.environ["OPENAI_API_KEY"] = userdata.get("OpenAI")
client = OpenAI()


# -------------------------------
# Load Dataset
# -------------------------------
with open("labeled_dataset_clean.json", "r") as f:
    data = json.load(f)


# -------------------------------
# Prompt Variants
# -------------------------------

def prompt_simple(text):
    return f"""
Score this essay (1–5):

1. Authenticity
2. Need
3. Desire

Return JSON only.

Essay:
{text[:1200]}
"""


def prompt_rubric(text):
    return f"""
You are an expert scholarship evaluator.

Score from 1 to 5:

Authenticity:
- 1 = not personal
- 5 = deeply personal and real

Need:
- 1 = no hardship
- 5 = clear financial/life hardship

Desire:
- 1 = no goals
- 5 = strong ambition

Be strict.

Return JSON only:
{{
  "authenticity": number,
  "need": number,
  "desire": number
}}

Essay:
{text[:1200]}
"""


def prompt_reasoning(text):
    return f"""
Evaluate the essay step by step:

1. Is it personal or academic?
2. Does it show hardship?
3. Does it show ambition?

Then give scores (1–5).

Return ONLY JSON:
{{
  "authenticity": number,
  "need": number,
  "desire": number
}}

Essay:
{text[:1200]}
"""


# -------------------------------
# LLM Call Function
# -------------------------------
def run_prompt(prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content


# -------------------------------
# JSON Fix
# -------------------------------
import re

def extract_json(text):
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        return match.group(0)
    return text


# -------------------------------
# Evaluate Prompts
# -------------------------------
results = []

NUM_TEST = 15  # test on subset

for i, record in enumerate(data[:NUM_TEST]):
    print(f"\nTesting Essay {i+1}")

    try:
        # Run all prompts
        out_a = run_prompt(prompt_simple(record["text"]))
        out_b = run_prompt(prompt_rubric(record["text"]))
        out_c = run_prompt(prompt_reasoning(record["text"]))

        # Parse
        res_a = json.loads(extract_json(out_a))
        res_b = json.loads(extract_json(out_b))
        res_c = json.loads(extract_json(out_c))

    except:
        print("⚠️ Error, skipping...")
        continue

    results.append({
        "text": record["text"],
        "human_auth": record["authenticity"],
        "human_need": record["need"],
        "human_desire": record["desire"],

        "A": res_a,
        "B": res_b,
        "C": res_c
    })


# Save results
with open("prompt_comparison.json", "w") as f:
    json.dump(results, f, indent=4)

print("✅ Prompt testing complete")


Testing Essay 1

Testing Essay 2

Testing Essay 3

Testing Essay 4

Testing Essay 5

Testing Essay 6

Testing Essay 7

Testing Essay 8

Testing Essay 9

Testing Essay 10

Testing Essay 11

Testing Essay 12

Testing Essay 13

Testing Essay 14

Testing Essay 15
✅ Prompt testing complete


NEXT: COMPARE PROMPTS

In [ ]:
print(data[0])

{'text': "The 'Face' on Mars is just a hoax. It seems as though the mound on this planet is just a mesa. If you don't believe me, then listen to me very closely. The very similar face on the side of Mars can be very defining but not a single astronout have researched at the Mesa up close and personal. But even if it was made by aliens, it still wouldn't tally up right? The 'Face' was first noticed in 1976 by Viking 1 spacecraft, circling around Mars, after trying to find its sistership, Viking 2. As Viking 1 was circling around, it spotted a likeness to the human face. And the mound was found in Cydonia, a very well-know area on the Red Planet to house mesas. As the years went by, they looked more into the mound and Scientists concluded that it was just another mesa, but this moutain-like hill had shadows that resemble the human face. As you review this very peculiar picture, you can see that it just resembles Earth, mostly in Western America. That certain part of America are very popu

In [ ]:
print(data[0].keys())

dict_keys(['text', 'human_auth', 'human_need', 'human_desire', 'A', 'B', 'C'])


In [ ]:
print(data[0]["A"])

{'score': 2, 'authenticity': 2, 'need': 2, 'desire': 3}


In [ ]:
import pandas as pd
import json

with open("prompt_comparison.json") as f:
    data = json.load(f)

rows = []

human_map = {
    "authenticity": "human_auth",
    "need": "human_need",
    "desire": "human_desire"
}

for d in data:
    for label in ["authenticity", "need", "desire"]:

        human_value = d[human_map[label]]

        for prompt in ["A", "B", "C"]:
            try:
                model_value = d[prompt][label]
            except:
                continue

            rows.append({
                "prompt": prompt,
                "diff": abs(model_value - human_value)
            })

df = pd.DataFrame(rows)

print(df.groupby("prompt")["diff"].mean())

prompt
A    0.777778
B    0.600000
C    1.000000
Name: diff, dtype: float64


#PHASE 10 — Baseline Models & Classical NLP Methods
## Description

Phase 10 implements traditional natural language processing (NLP) evaluation methods to serve as baselines for comparison against the LLM-based evaluation system. These methods include lexical overlap metrics such as BLEU, n-gram similarity, and simple heuristic-based scoring techniques.

The purpose of this phase is to assess how well classical approaches capture human-centered qualities like authenticity, need, and motivation. By comparing these baseline methods to LLM outputs, the project demonstrates the limitations of traditional metrics in evaluating nuanced, subjective text.

##Summary

Phase 10:

* implements BLEU and n-gram-based scoring,
* builds simple heuristic baselines, and
* prepares results for comparison with LLM and human evaluations.

In [ ]:
!pip install nltk scikit-learn

In [ ]:
# ===============================
# PHASE 10: BASELINE MODELS
# ===============================

import json
import pandas as pd
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np


# -------------------------------
# Load Dataset
# -------------------------------
with open("evaluated_dataset.json", "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print(f"Loaded {len(df)} samples.")


# -------------------------------
# BLEU SCORE (REFERENCE-BASED)
# -------------------------------
def compute_bleu(reference, candidate):
    ref_tokens = reference.split()
    cand_tokens = candidate.split()

    smoothie = SmoothingFunction().method1
    return sentence_bleu([ref_tokens], cand_tokens, smoothing_function=smoothie)


# -------------------------------
# N-GRAM SIMILARITY
# -------------------------------
vectorizer = CountVectorizer(ngram_range=(1,2))


def compute_ngram_similarity(text1, text2):
    vectors = vectorizer.fit_transform([text1, text2]).toarray()

    v1, v2 = vectors[0], vectors[1]

    # cosine similarity
    if np.linalg.norm(v1) == 0 or np.linalg.norm(v2) == 0:
        return 0

    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))


# -------------------------------
# BASELINE SCORING FUNCTION
# -------------------------------
def baseline_score(text):
    text_lower = text.lower()

    # simple keyword-based scoring
    authenticity = 1
    need = 1
    desire = 1

    if "i " in text_lower or "my " in text_lower:
        authenticity += 1

    if any(word in text_lower for word in ["family", "struggle", "money", "poor"]):
        need += 1

    if any(word in text_lower for word in ["dream", "goal", "future", "aspire"]):
        desire += 1

    return {
        "baseline_authenticity": min(authenticity, 5),
        "baseline_need": min(need, 5),
        "baseline_desire": min(desire, 5)
    }


# -------------------------------
# APPLY BASELINES
# -------------------------------
baseline_results = []

for i, row in df.iterrows():
    text = row["text"]

    base = baseline_score(text)

    row["baseline_authenticity"] = base["baseline_authenticity"]
    row["baseline_need"] = base["baseline_need"]
    row["baseline_desire"] = base["baseline_desire"]

    # Compare with LLM output (optional)
    bleu = compute_bleu(text, text)  # trivial baseline
    row["bleu_score"] = bleu

    baseline_results.append(row)


# -------------------------------
# Save Results
# -------------------------------
df_out = pd.DataFrame(baseline_results)

df_out.to_json("baseline_results.json", orient="records", indent=4)

print("✅ Baseline evaluation complete.")

Loaded 21 samples.
✅ Baseline evaluation complete.


#PHASE 11 — Final Comparison & Evaluation
## Description

Phase 11 evaluates and compares the performance of all methods implemented in the project, including human annotations (ground truth), LLM-based evaluation, and classical baseline approaches.

This phase quantifies alignment between methods using error metrics and summarizes findings to determine which approach best captures human-centered evaluation criteria such as authenticity, need, and motivation.

##Summary

Phase 11:

* compares human vs LLM vs baseline
* computes error metrics
* identifies best-performing method
* produces final results for your report

In [ ]:
# ===============================
# PHASE 11: FINAL COMPARISON
# ===============================

import json
import pandas as pd
import numpy as np


# -------------------------------
# Load Data
# -------------------------------
with open("evaluated_dataset.json", "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print(f"Loaded {len(df)} samples.")


# -------------------------------
# ERROR FUNCTION
# -------------------------------
def compute_error(true, pred):
    return abs(true - pred)


# -------------------------------
# CALCULATE ERRORS
# -------------------------------
results = []

for _, row in df.iterrows():
    for label in ["authenticity", "need", "desire"]:

        human = row[label]
        llm = row.get(f"llm_{label}")
        baseline = row.get(f"baseline_{label}")

        if llm is not None:
            results.append({
                "method": "LLM",
                "label": label,
                "error": compute_error(human, llm)
            })

        if baseline is not None:
            results.append({
                "method": "Baseline",
                "label": label,
                "error": compute_error(human, baseline)
            })


df_results = pd.DataFrame(results)


# -------------------------------
# OVERALL PERFORMANCE
# -------------------------------
print("\n=== OVERALL ERROR ===")
print(df_results.groupby("method")["error"].mean())


# -------------------------------
# PER-LABEL PERFORMANCE
# -------------------------------
print("\n=== ERROR BY LABEL ===")
print(df_results.groupby(["method", "label"])["error"].mean())


# -------------------------------
# BEST METHOD
# -------------------------------
best_method = df_results.groupby("method")["error"].mean().idxmin()

print(f"\n🏆 Best performing method: {best_method}")

Loaded 21 samples.

=== OVERALL ERROR ===
method
LLM    0.714286
Name: error, dtype: float64

=== ERROR BY LABEL ===
method  label       
LLM     authenticity    0.333333
        desire          0.952381
        need            0.857143
Name: error, dtype: float64

🏆 Best performing method: LLM


#PHASE 12 — Bias Analysis (FULL)
##Description

Phase 12 investigates whether the LLM evaluation system exhibits systematic biases when scoring essays. Specifically, it examines whether non-semantic factors—such as essay length, vocabulary richness, or stylistic features—unduly influence scores.

The goal is to ensure that the model evaluates essays based on meaningful content rather than superficial characteristics.

##Summary

Phase 12:

* analyzes correlation between essay features and scores,
* detects potential biases (e.g., length bias), and
* validates fairness of the evaluation system.

In [ ]:
# ===============================
# PHASE 12: BIAS ANALYSIS
# ===============================

import pandas as pd
import json
import numpy as np


# -------------------------------
# Load Dataset
# -------------------------------
with open("evaluated_dataset.json", "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print(f"Loaded {len(df)} samples.")


# -------------------------------
# FEATURE ENGINEERING
# -------------------------------

# Essay length
df["length"] = df["text"].apply(len)

# Word count
df["word_count"] = df["text"].apply(lambda x: len(x.split()))

# Avg word length (proxy for vocabulary complexity)
df["avg_word_length"] = df["text"].apply(
    lambda x: np.mean([len(w) for w in x.split()]) if len(x.split()) > 0 else 0
)

# Simple emotion proxy
emotion_words = ["love", "struggle", "pain", "dream", "hope", "family"]

df["emotion_score"] = df["text"].apply(
    lambda x: sum(1 for w in emotion_words if w in x.lower())
)


# -------------------------------
# CORRELATION ANALYSIS
# -------------------------------
print("\n=== CORRELATION WITH LLM SCORES ===")

features = ["length", "word_count", "avg_word_length", "emotion_score"]
labels = ["llm_authenticity", "llm_need", "llm_desire"]

for label in labels:
    print(f"\n--- {label.upper()} ---")
    for feature in features:
        corr = df[feature].corr(df[label])
        print(f"{feature}: {round(corr, 3)}")


# -------------------------------
# HUMAN VS LLM BIAS COMPARISON
# -------------------------------
print("\n=== HUMAN VS LLM LENGTH BIAS ===")

for label in ["authenticity", "need", "desire"]:
    human_corr = df["length"].corr(df[label])
    llm_corr = df["length"].corr(df[f"llm_{label}"])

    print(f"\n{label.upper()}")
    print(f"Human correlation: {round(human_corr, 3)}")
    print(f"LLM correlation:   {round(llm_corr, 3)}")


# -------------------------------
# GROUP ANALYSIS (SHORT VS LONG)
# -------------------------------
print("\n=== SHORT vs LONG ESSAYS ===")

median_length = df["length"].median()

short = df[df["length"] <= median_length]
long = df[df["length"] > median_length]

for label in ["llm_authenticity", "llm_need", "llm_desire"]:
    print(f"\n{label}")
    print("Short avg:", round(short[label].mean(), 2))
    print("Long avg:", round(long[label].mean(), 2))


# -------------------------------
# SUMMARY
# -------------------------------
print("\n=== BIAS ANALYSIS COMPLETE ===")

Loaded 21 samples.

=== CORRELATION WITH LLM SCORES ===

--- LLM_AUTHENTICITY ---
length: 0.049
word_count: 0.058
avg_word_length: -0.091
emotion_score: -0.188

--- LLM_NEED ---
length: -0.289
word_count: -0.294
avg_word_length: -0.116
emotion_score: -0.194

--- LLM_DESIRE ---
length: -0.241
word_count: -0.242
avg_word_length: -0.154
emotion_score: -0.078

=== HUMAN VS LLM LENGTH BIAS ===

AUTHENTICITY
Human correlation: -0.246
LLM correlation:   0.049

NEED
Human correlation: 0.249
LLM correlation:   -0.289

DESIRE
Human correlation: -0.18
LLM correlation:   -0.241

=== SHORT vs LONG ESSAYS ===

llm_authenticity
Short avg: 1.27
Long avg: 1.3

llm_need
Short avg: 1.09
Long avg: 1.0

llm_desire
Short avg: 1.73
Long avg: 1.2

=== BIAS ANALYSIS COMPLETE ===


#PHASE 13 — Hallucination & Robustness Testing
## Description

Phase 13 evaluates the robustness of the LLM evaluation system by testing its behavior on controlled inputs, including weak, synthetic, and adversarial essays. The goal is to determine whether the model assigns inflated scores to low-quality content or hallucinates depth where none exists.

This phase helps assess whether the system is reliable in real-world scenarios where inputs may be misleading, vague, or artificially constructed.

## Summary

Phase 13:

* tests the model on weak and synthetic essays,
* evaluates susceptibility to hallucination, and
* measures robustness against misleading inputs.

In [ ]:
# ===============================
# PHASE 13: HALLUCINATION & ROBUSTNESS
# ===============================

import json

# reuse your function from Phase 8
# evaluate_with_llm(text)


# -------------------------------
# TEST CASES
# -------------------------------
test_cases = [
    {
        "type": "weak",
        "text": "I want to succeed. I am hardworking. I believe in myself."
    },
    {
        "type": "generic",
        "text": "Education is important. Many people want success. It is good to have goals."
    },
    {
        "type": "repetitive",
        "text": "I want success. I want success. I want success. I want success."
    },
    {
        "type": "emotional_fake",
        "text": "My life has been very hard. I struggled so much. Everything was difficult. I want a better future."
    },
    {
        "type": "nonsense",
        "text": "Blue sky run fast apple dream logic random thinking world jump."
    }
]


# -------------------------------
# RUN TESTS
# -------------------------------
results = []

for case in test_cases:
    print("\n" + "="*80)
    print(f"Test Type: {case['type']}")
    print("="*80)

    print("\nText:")
    print(case["text"])

    try:
        output = evaluate_with_llm(case["text"])
        print("\n🤖 LLM Output:")
        print(output)

        parsed = extract_json(output)

    except Exception as e:
        print("⚠️ Error:", e)
        continue

    results.append({
        "type": case["type"],
        "text": case["text"],
        "result": parsed
    })


# -------------------------------
# SAVE RESULTS
# -------------------------------
with open("hallucination_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("\n✅ Hallucination testing complete.")


Test Type: weak

Text:
I want to succeed. I am hardworking. I believe in myself.

🤖 LLM Output:
{
  "authenticity": 1,
  "need": 1,
  "desire": 2
}

Test Type: generic

Text:
Education is important. Many people want success. It is good to have goals.

🤖 LLM Output:
{
  "authenticity": 1,
  "need": 1,
  "desire": 1
}

Test Type: repetitive

Text:
I want success. I want success. I want success. I want success.

🤖 LLM Output:
{
  "authenticity": 1,
  "need": 1,
  "desire": 2
}

Test Type: emotional_fake

Text:
My life has been very hard. I struggled so much. Everything was difficult. I want a better future.

🤖 LLM Output:
{
  "authenticity": 1,
  "need": 1,
  "desire": 2
}

Test Type: nonsense

Text:
Blue sky run fast apple dream logic random thinking world jump.

🤖 LLM Output:
{
  "authenticity": 1,
  "need": 1,
  "desire": 1
}

✅ Hallucination testing complete.


In [ ]:
for r in results:
    print(r["type"], r["result"])

weak {
  "authenticity": 1,
  "need": 1,
  "desire": 2
}
generic {
  "authenticity": 1,
  "need": 1,
  "desire": 1
}
repetitive {
  "authenticity": 1,
  "need": 1,
  "desire": 2
}
emotional_fake {
  "authenticity": 1,
  "need": 1,
  "desire": 2
}
nonsense {
  "authenticity": 1,
  "need": 1,
  "desire": 1
}


#PHASE 14 — Multi-Model Comparison
## Description

Phase 14 evaluates how different large language models perform on the same essay scoring task. By comparing outputs from multiple models, this phase assesses consistency, accuracy, and reliability across architectures.

The goal is to determine whether model choice significantly impacts performance in human-centered evaluation tasks such as authenticity, need, and motivation.

## Summary

Phase 14:

* evaluates multiple LLMs on the same dataset,
* compares their outputs against human labels, and
* identifies performance differences across models.

In [ ]:
# ===============================
# PHASE 14: MULTI-MODEL COMPARISON (FULL FIXED)
# ===============================

# -------------------------------
# Setup (if not already)
# -------------------------------
from google.colab import userdata
import os
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = userdata.get("OpenAI")
client = OpenAI()

# -------------------------------
# Imports
# -------------------------------
import json
import re
import pandas as pd


# -------------------------------
# Load Dataset
# -------------------------------
with open("labeled_dataset_clean.json", "r") as f:
    data = json.load(f)

print(f"Loaded {len(data)} samples.")


# -------------------------------
# Robust JSON Parser
# -------------------------------
def extract_json(text):
    try:
        return json.loads(text)
    except:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except:
                return None
    return None


# -------------------------------
# Prompt
# -------------------------------
def build_prompt(text):
    return f"""
You are an expert scholarship evaluator.

Score from 1 to 5:

1. Authenticity
2. Need
3. Desire

Return ONLY JSON:
{{
  "authenticity": number,
  "need": number,
  "desire": number
}}

Essay:
{text[:1000]}
"""


# -------------------------------
# Models to Compare
# -------------------------------
models = [
    {"name": "gpt4o-mini", "model": "gpt-4o-mini"},
    {"name": "gpt4o", "model": "gpt-4o"}  # optional
]


# -------------------------------
# Run Evaluation
# -------------------------------
results = []

for i, record in enumerate(data[:15]):  # limit for cost
    print("\n" + "="*70)
    print(f"Sample {i+1}")
    print("="*70)

    for m in models:
        print(f"\n🔹 Running model: {m['name']}")

        try:
            response = client.chat.completions.create(
                model=m["model"],
                messages=[{"role": "user", "content": build_prompt(record["text"])}],
                temperature=0
            )

            output = response.choices[0].message.content

            print("RAW OUTPUT:", output)

            parsed = extract_json(output)

            if parsed is None:
                print("⚠️ Failed to parse JSON")
                continue

        except Exception as e:
            print(f"⚠️ Error with {m['name']}:", e)
            continue

        # -------------------------------
        # SAFE VALUE EXTRACTION
        # -------------------------------
        llm_auth = parsed.get("authenticity")
        llm_need = parsed.get("need")
        llm_desire = parsed.get("desire")

        # Skip if missing values
        if llm_auth is None or llm_need is None or llm_desire is None:
            print("⚠️ Missing keys, skipping...")
            continue

        results.append({
            "model": m["name"],
            "text": record["text"],
            "human_auth": record["authenticity"],
            "human_need": record["need"],
            "human_desire": record["desire"],
            "llm_auth": llm_auth,
            "llm_need": llm_need,
            "llm_desire": llm_desire
        })


# -------------------------------
# Save Results
# -------------------------------
with open("multi_model_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("\n✅ Multi-model evaluation complete.")

Loaded 21 samples.

Sample 1

🔹 Running model: gpt4o-mini
RAW OUTPUT: {
  "authenticity": 1,
  "need": 1,
  "desire": 1
}

🔹 Running model: gpt4o
RAW OUTPUT: ```json
{
  "authenticity": 1,
  "need": 1,
  "desire": 1
}
```

Sample 2

🔹 Running model: gpt4o-mini
RAW OUTPUT: {
  "authenticity": 4,
  "need": 3,
  "desire": 5
}

🔹 Running model: gpt4o
RAW OUTPUT: ```json
{
  "authenticity": 4,
  "need": 3,
  "desire": 4
}
```

Sample 3

🔹 Running model: gpt4o-mini
RAW OUTPUT: {
  "authenticity": 3,
  "need": 4,
  "desire": 4
}

🔹 Running model: gpt4o
RAW OUTPUT: ```json
{
  "authenticity": 3,
  "need": 2,
  "desire": 4
}
```

Sample 4

🔹 Running model: gpt4o-mini
RAW OUTPUT: {
  "authenticity": 3,
  "need": 2,
  "desire": 4
}

🔹 Running model: gpt4o
RAW OUTPUT: ```json
{
  "authenticity": 3,
  "need": 3,
  "desire": 4
}
```

Sample 5

🔹 Running model: gpt4o-mini
RAW OUTPUT: ```json
{
  "authenticity": 3,
  "need": 2,
  "desire": 4
}
```

🔹 Running model: gpt4o
RAW OUTPUT: ```json
{
  "authe

#PHASE 15 — Error Analysis & Case Studies
## Description

Phase 15 conducts a detailed qualitative analysis of model performance by examining specific cases where the LLM evaluation diverges from human judgment. This phase identifies patterns in model errors, such as misunderstanding context, overvaluing emotional language, or missing implicit meaning.

The goal is to provide insight into why the model succeeds or fails, complementing the quantitative results from earlier phases.

## Summary

Phase 15:

* identifies high-error samples,
* analyzes model failures, and
* provides case studies to explain result

In [ ]:
# ===============================
# PHASE 15: ERROR ANALYSIS
# ===============================

import pandas as pd
import json


# -------------------------------
# Load Evaluated Dataset
# -------------------------------
with open("evaluated_dataset.json", "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print(f"Loaded {len(df)} samples.")


# -------------------------------
# Compute Errors
# -------------------------------
df["error_auth"] = abs(df["authenticity"] - df["llm_authenticity"])
df["error_need"] = abs(df["need"] - df["llm_need"])
df["error_desire"] = abs(df["desire"] - df["llm_desire"])

df["total_error"] = df[["error_auth", "error_need", "error_desire"]].mean(axis=1)


# -------------------------------
# WORST CASES
# -------------------------------
worst_cases = df.sort_values("total_error", ascending=False).head(5)

print("\n=== WORST CASES ===")

for i, row in worst_cases.iterrows():
    print("\n" + "="*80)
    print("TEXT:")
    print(row["text"][:500])

    print("\nHUMAN SCORES:")
    print({
        "auth": row["authenticity"],
        "need": row["need"],
        "desire": row["desire"]
    })

    print("\nLLM SCORES:")
    print({
        "auth": row["llm_authenticity"],
        "need": row["llm_need"],
        "desire": row["llm_desire"]
    })

    print("\nERROR:", round(row["total_error"], 2))


# -------------------------------
# BEST CASES
# -------------------------------
best_cases = df.sort_values("total_error", ascending=True).head(5)

print("\n=== BEST CASES ===")

for i, row in best_cases.iterrows():
    print("\n" + "="*80)
    print("TEXT:")
    print(row["text"][:500])

    print("\nHUMAN:", row["authenticity"], row["need"], row["desire"])
    print("LLM:", row["llm_authenticity"], row["llm_need"], row["llm_desire"])

    print("ERROR:", round(row["total_error"], 2))


# -------------------------------
# ERROR PATTERNS
# -------------------------------
print("\n=== ERROR PATTERNS ===")

print("\nAverage Errors:")
print(df[["error_auth", "error_need", "error_desire"]].mean())

Loaded 21 samples.

=== WORST CASES ===

TEXT:
Luke was a great guy he was a Seagoing Cowboy who worked 2 jobs so as, grocery store and bank. He thought things wouldn't be well after he got done with his graduation. Boom! Something changed, it was a opportunity for him so let me tell you. One day his frend Don asked him do u want to go on a Europe cattle boat with me. Luke said yes because he thught it would been a nice thing to do. After that Luke and Don seen that the UNRRA were looking for Seagoing Cowboys to help out with animals that we

HUMAN SCORES:
{'auth': 5.0, 'need': 1.0, 'desire': 3.0}

LLM SCORES:
{'auth': 1, 'need': 1, 'desire': 1}

ERROR: 2.0

TEXT:
We have been fascinated on driverless cars since the late 1950s.There is both pros and cons when it comes to the aspects of driverless cars. Sergey the google cofunder envisions a futeer with a public transportation sysytem theat would use half the fuel of a taxi .The car would even have more flexability than a bus .driverles